# Pré-processamento
### 1. Leitura do dataset

In [47]:
import pandas as pd

# Login using e.g. `huggingface-cli login` to access this dataset
splits = {'train': 'train.csv', 'test': 'test.csv'}
df_treino = pd.read_csv("hf://datasets/c2d-usp/banking77-pt-br/" + splits["train"])
df_teste = pd.read_csv("hf://datasets/c2d-usp/banking77-pt-br/" + splits["test"])

df = pd.concat([df_treino,df_teste], ignore_index=True)

# Registro de quantas linhas restam após cada etapa (resumo no final do notebook)
etapas = {'Dataset original': len(df)}

display(df.shape)
df.head(10)

(13085, 4)

,frase,rotulo,modelo_tradutor,alterado
0,Ainda estou esperando pelo meu cartão?,chegada_do_cartão,gpt 3.5-turbo,0
1,O que posso fazer se meu cartão ainda não chegou depois de 2 semanas?,chegada_do_cartão,gpt 3.5-turbo,0
2,Eu tenho esperado por mais de uma semana. O cartão ainda vai chegar?,chegada_do_cartão,gpt 3.5-turbo,0
3,Posso rastrear meu cartão enquanto está em processo de entrega?,chegada_do_cartão,gpt 3.5-turbo,0
4,"Como eu posso saber se vou receber meu cartão, ou se ele está perdido?",chegada_do_cartão,gpt 3.5-turbo,0
5,Quando você me enviou meu novo cartão?,chegada_do_cartão,gpt 3.5-turbo,0
6,Você tem informações sobre a entrega do cartão?,chegada_do_cartão,gpt 3.5-turbo,1
7,O que devo fazer se ainda não recebi meu novo cartão?,chegada_do_cartão,gpt 3.5-turbo,0
8,O pacote com meu cartão tem rastreamento?,chegada_do_cartão,gpt 3.5-turbo,0
9,"Eu pedi meu cartão, mas ele ainda não chegou.",chegada_do_cartão,gpt 3.5-turbo,0


In [48]:
for i in df.columns:
    display(df[i].unique())


<ArrowStringArray>
[                                'Ainda estou esperando pelo meu cartão?',
  'O que posso fazer se meu cartão ainda não chegou depois de 2 semanas?',
   'Eu tenho esperado por mais de uma semana. O cartão ainda vai chegar?',
        'Posso rastrear meu cartão enquanto está em processo de entrega?',
 'Como eu posso saber se vou receber meu cartão, ou se ele está perdido?',
                                 'Quando você me enviou meu novo cartão?',
                        'Você tem informações sobre a entrega do cartão?',
                  'O que devo fazer se ainda não recebi meu novo cartão?',
                              'O pacote com meu cartão tem rastreamento?',
                          'Eu pedi meu cartão, mas ele ainda não chegou.',
 ...
                                     'Em quais países você faz negócios?',
               'Eu não moro no Reino Unido. Ainda posso obter um cartão?',
                               'Em quais lugares vocês oferecem suporte?',
 

<ArrowStringArray>
[                                       'chegada_do_cartão',
                                     'vinculação_de_cartão',
                                           'taxa_de_câmbio',
                   'pagamento_cartão_taxa_câmbio_incorreta',
                                'cobrança_extra_no_extrato',
                               'saque_de_dinheiro_pendente',
                                 'suporte_moeda_fiduciária',
                          'estimativa_de_entrega_do_cartão',
                                       'recarga_automática',
                                   'cartão_não_funcionando',
                                     'troca_via_aplicativo',
                                'cartão_perdido_ou_roubado',
                                          'limite_de_idade',
                                            'pin_bloqueado',
                              'sem_contato_não_funcionando',
              'recarga_por_transferência_bancária_cobrança',
     

<ArrowStringArray>
['gpt 3.5-turbo', 'gpt 4o-mini', nan]
Length: 3, dtype: str

array([0, 1])

`pagamento_cartão_revertido?` está com '?' no final. Correção:

In [49]:
df['rotulo'] = df['rotulo'].str.rstrip('?')

#### 1.1 Remoção de colunas sem utilidade

Removo as colunas referentes a um indicador de revisão da tradução, e o modelo que o ChatGPT fez a tradução, ambos sem informações relevantes para o modelo

In [50]:
df = df.drop(columns=['alterado', 'modelo_tradutor'])

#### 1.2 Valores nulos

In [51]:
df.isna().sum()

frase     2
rotulo    2
dtype: int64

In [52]:
df[df['frase'].isna()]

,frase,rotulo
13083,NaN,NaN
13084,NaN,NaN


In [53]:
df = df.dropna(subset=['frase'])
etapas['Valores nulos'] = len(df)

#### 1.3 Tratamento dos textos

O dataset foi traduzido do inglês com o ChatGPT `gpt 3.5-turbo` e `gpt 4o-mini` e depois revisado. Portanto, é necessário uma análise para ver se não ficaram  resquícios de erros e sujeiras nos textos.

Primeiramente, suspeitei de textos com aspas no início e no final

In [54]:
df[df['frase'].str.contains('"')]

,frase,rotulo
388,"""Quais são as suas taxas de câmbio?""",taxa_de_câmbio
694,"Estou bastante frustrado com uma cobrança de libra esterlina que continua aparecendo no extrato da minha conta. Está listada no aplicativo como ""pendente"", nunca muda e agora estou pensando que alguém invadiu minha conta!",cobrança_extra_no_extrato
783,"O que significa ""pendente"" em um saque de dinheiro?",saque_de_dinheiro_pendente
788,"A máquina não funcionou mais cedo quando fui sacar dinheiro e agora minha conta diz ""transação em andamento"". Por quê?",saque_de_dinheiro_pendente
1560,"""Meus filhos podem abrir uma conta?""",limite_de_idade
1569,"Do you want me to translate the text ""What is needed for my daughter to open an account?"" to Brazilian Portuguese?",limite_de_idade
1580,"""O que minha filha precisa para abrir uma conta?""",limite_de_idade
1581,"""Meus filhos podem ter uma conta?""",limite_de_idade
1597,"""Posso abrir uma conta para meus filhos?""",limite_de_idade
1610,"""Meus filhos podem ter uma conta?""",limite_de_idade


In [55]:
df['frase'] = df['frase'].str.strip().str.strip('"“”\'')

Em seguida, para uma análise mais profunda (sem padrão claro), utilizei IA (`Claude Code Opus 5.5`) para explorar os valores para ver se existiam problemas de tradução direta do inglês para o português, ou sujeiras específicas do ChatGPT.

Categorias encontradas:

- **Não é frase de cliente:** o lugar da tradução foi ocupado por uma resposta do próprio GPT (*"Do you want me to translate the text..."*) ou por um comentário do revisor (*"Tradução com problema mínimo"*). Essas linhas serão **removidas**.
- **Anotação do revisor:** a frase está correta, mas tem um comentário colado no fim (*"/ Observação: Frase estranha"*). A anotação será **cortada** e a frase mantida. Num desses casos o comentário mostra também um erro de tradução: o original tinha *"car"* digitado no lugar de *"card"*, e o GPT traduziu *top up* como "abastecer".
- **Marcador não preenchido:** um marcador como `[país]` ficou no texto sem ser substituído. Ele será **substituído** por um termo genérico.

In [56]:
pd.set_option('display.max_colwidth', None)

s = df['frase']
problemas = {
    'Não é frase de cliente':  s.str.contains(r'^Tradução com problema|translate the text', case=False),
    'Anotação do revisor':     s.str.contains(r'\s*/\s*Observa[cç][aã]o:', case=False),
    'Marcador não preenchido': s.str.contains(r'\[[^\]]+\]'),
}

casos = pd.concat(
    [df.loc[m, ['frase', 'rotulo']].assign(problema=nome) for nome, m in problemas.items()]
)

display(casos['problema'].value_counts())
display(casos)

problema
Anotação do revisor        4
Não é frase de cliente     2
Marcador não preenchido    1
Name: count, dtype: int64

,frase,rotulo,problema
71,Tradução com problema mínimo,chegada_do_cartão,Não é frase de cliente
1569,"Do you want me to translate the text ""What is needed for my daughter to open an account?"" to Brazilian Portuguese?",limite_de_idade,Não é frase de cliente
7350,"Meu salário está em GBP - o que devo fazer? / Observação: Uso da expressão ""libras esterlinas"" ao invés da sigla GBP",receber_dinheiro,Anotação do revisor
7370,Como posso saber qual a configuração da minha conta? Eu sou pago em GBP. /Observação: Frase estranha,receber_dinheiro,Anotação do revisor
7637,"Para minha conta, como eu transfiro dinheiro? / Observação: Tradução não realizada",transferir_para_conta,Anotação do revisor
8533,"Tentei abastecer meu cartão hoje, mas falhou! Eu o comprei há apenas alguns dias e funcionou então. O que está acontecendo com ele? / Observação: escreveu 'car' ao invés de 'card",recarga_falha,Anotação do revisor
12716,Onde está minha transferência de [país]?,saldo_não_atualizado_após_transferência_bancária,Marcador não preenchido


In [57]:
# 1. remover linhas que não são frases de cliente
nao_frase = df['frase'].str.contains(r'^Tradução com problema|translate the text', case=False)
df = df[~nao_frase].copy()

# 2. cortar anotações "/ Observação: ..."
df['frase'] = df['frase'].str.replace(r'\s*/\s*Observa[cç][aã]o:.*$', '', regex=True)

# 3. corrigir tradução errada ("top up my car" virou "abastecer")
df['frase'] = df['frase'].str.replace('Tentei abastecer meu cartão', 'Tentei recarregar meu cartão', regex=False)

# 4. preencher o marcador
df['frase'] = df['frase'].str.replace('de [país]', 'do exterior', regex=False)

etapas['Limpeza da tradução'] = len(df)

##### Erro sistemático: "top up" traduzido como "abastecer"

Na análise do Claude Code, também foi encontrado o seguinte caso:

Em inglês, *top up* significa recarregar a conta ou o cartão, mas o GPT entendeu como abastecer o carro: "abastecer meu tanque de gasolina", "pagar o combustível", "reabastecimento". Em algumas frases o original ainda tinha *car* digitado no lugar de *card*, o que virou "carro".

São 16 frases, quase todas das classes de recarga e de Apple Pay. Isso atrapalharia o modelo, porque palavras como "gasolina" e "tanque" passariam a indicar recarga.

As frases foram corrigidas uma a uma. As que falam de carro ou combustível de verdade foram mantidas: cartão roubado do carro, aluguel de carro, pagamento num posto.

Também foram feitos dois ajustes menores: a frase `???`, que não tem conteúdo, foi removida, e o rótulo `pagamento_cartão_revertido?` perdeu o `?` do nome.

In [58]:
termos_carro = r'(?i)abastec|gasolina|combust[ií]vel|tanque|\bcarro\b'
display(df.loc[df['frase'].str.contains(termos_carro), ['frase', 'rotulo']])

,frase,rotulo
1954,"Onde estão os meus fundos? Abasteci o meu carro, mas parece que a transação não foi concluída.",recarga_pendente
2060,"novo cliente e abastecendo o tanque pela primeira vez hoje, não funcionando.",recarga_pendente
4478,O que posso fazer se os detalhes do meu cartão foram roubados do meu carro? Acho que usaram meu cartão para comprar gasolina.,cartão_comprometido
5474,"Eu não quero mais comprar este carro, posso cancelar a transação e receber um reembolso rápido?",solicitar_reembolso
5970,Por que os pagamentos estão sendo recusados no meu cartão novo? Tentei várias vezes usar meu cartão novo para pagar combustível no posto Shell da minha vizinhança. Estou muito aflito porque meu cartão novo não está funcionando.,pagamento_do_cartão_recusado
7777,Onde encontro o código de verificação do cartão que estou reabastecendo?,verificar_recarga
7939,Posso usar um cheque para abastecer?,recarregar_por_dinheiro_ou_cheque
8026,Como posso abastecer meu tanque de gasolina usando um cheque?,recarregar_por_dinheiro_ou_cheque
8188,Meu carro virtual não está funcionando.,cartão_virtual_não_funcionando
8207,Meu cartão virtual foi rejeitado para o aluguel de um carro. Por quê?,cartão_virtual_não_funcionando


In [59]:
correcoes = {
    'Onde estão os meus fundos? Abasteci o meu carro, mas parece que a transação não foi concluída.':
        'Onde estão os meus fundos? Recarreguei o meu cartão, mas parece que a transação não foi concluída.',
    'novo cliente e abastecendo o tanque pela primeira vez hoje, não funcionando.':
        'novo cliente e fazendo a recarga pela primeira vez hoje, não está funcionando.',
    'Onde encontro o código de verificação do cartão que estou reabastecendo?':
        'Onde encontro o código de verificação do cartão que estou usando na recarga?',
    'Posso usar um cheque para abastecer?':
        'Posso usar um cheque para recarregar?',
    'Como posso abastecer meu tanque de gasolina usando um cheque?':
        'Como posso fazer uma recarga usando um cheque?',
    'Meu carro virtual não está funcionando.':
        'Meu cartão virtual não está funcionando.',
    'Meu reabastecimento foi negado.':
        'Minha recarga foi negada.',
    'Posso usar meu Apple Watch para pagar o combustível?':
        'Posso usar meu Apple Watch para fazer uma recarga?',
    'O reabastecimento não está funcionando, embora eu tenha meu AMEX no Apple Pay.':
        'A recarga não está funcionando, embora eu tenha meu AMEX no Apple Pay.',
    'Posso usar meu Apple Watch para abastecer meu tanque de gasolina?':
        'Posso usar meu Apple Watch para recarregar minha conta?',
    'Posso pagar meu combustível usando meu Apple Watch?':
        'Posso fazer a recarga usando meu Apple Watch?',
    'Você pode, por favor, fazer meu reabastecimento ser processado o mais rápido possível? Eu realmente preciso do dinheiro e já está pendente há uma hora.':
        'Você pode, por favor, fazer minha recarga ser processada o mais rápido possível? Eu realmente preciso do dinheiro e já está pendente há uma hora.',
    'Posso abastecer usando meu carro?':
        'Posso recarregar usando meu cartão?',
    'Onde eu localizo o abastecimento com dinheiro?':
        'Onde eu faço a recarga com dinheiro?',
    'Meu reabastecimento falhou, POR QUÊ?.':
        'Minha recarga falhou, POR QUÊ?',
    'Há uma taxa ou desconto se eu usar um banco europeu em um reabastecimento?':
        'Há uma taxa ou desconto se eu usar um banco europeu em uma recarga?',
}
print(f"Linhas corrigidas: {df['frase'].isin(correcoes.keys()).sum()}")
df['frase'] = df['frase'].replace(correcoes)

# Conferência: o que sobrou com esses termos fala de carro/combustível de verdade
print('Frases restantes com termos de carro/combustível:')
display(df.loc[df['frase'].str.contains(termos_carro), ['frase', 'rotulo']])

# Frase sem conteúdo
df = df[df['frase'] != '???'].copy()

etapas['Correção de "abastecer" e frase vazia'] = len(df)

Linhas corrigidas: 16
Frases restantes com termos de carro/combustível:


,frase,rotulo
4478,O que posso fazer se os detalhes do meu cartão foram roubados do meu carro? Acho que usaram meu cartão para comprar gasolina.,cartão_comprometido
5474,"Eu não quero mais comprar este carro, posso cancelar a transação e receber um reembolso rápido?",solicitar_reembolso
5970,Por que os pagamentos estão sendo recusados no meu cartão novo? Tentei várias vezes usar meu cartão novo para pagar combustível no posto Shell da minha vizinhança. Estou muito aflito porque meu cartão novo não está funcionando.,pagamento_do_cartão_recusado
8207,Meu cartão virtual foi rejeitado para o aluguel de um carro. Por quê?,cartão_virtual_não_funcionando
10999,Os postos de combustível aceitarão meu cartão?,aceitação_de_cartão


#### 1.4 Valores duplicados

In [60]:
dup_tupla = df.duplicated(subset=['frase', 'rotulo'])
dup_frase = df.duplicated(subset=['frase'])
print(dup_tupla.sum(), dup_frase.sum())

438 458


Como a quantidade de frases duplicadas é diferente da quantidade de tuplas (frase, rotulo) duplicadas, significa que existem frases no dataset que aparecem mais de uma vez, mas com rótulos diferentes. Antes de remover todos os casos de duplicadas em `frase` no geral, vou remover primeiro esses casos ambíguos para não deixar frases com rótulos possivelmente subjetivos

In [61]:
n_rotulos = df.groupby('frase')['rotulo'].nunique()
conflitos = n_rotulos[n_rotulos > 1].index
print(f'Frases com mais de um rótulo: {len(conflitos)}')
print('Número de casos ', len(df) - len(df[~df['frase'].isin(conflitos)]))

Frases com mais de um rótulo: 19
Número de casos  53


Um exemplo abaixo

In [62]:
df[df['frase'].str.contains('A taxa de câmbio muda?', na=False)]

,frase,rotulo
363,A taxa de câmbio muda?,taxa_de_câmbio
472,A taxa de câmbio muda?,pagamento_cartão_taxa_câmbio_incorreta


In [63]:
print('Todos os casos')
display(df[df['frase'].isin(conflitos)].groupby(['frase', 'rotulo']).size().to_frame('ocorrências'))

Todos os casos


ocorrências
frase                                                rotulo                                                       
A taxa de câmbio muda?                               pagamento_cartão_taxa_câmbio_incorreta                      1
                                                     taxa_de_câmbio                                              1
Há uma taxa de câmbio?                               cobrança_de_câmbio                                          1
                                                     taxa_de_câmbio                                              1
Minha recarga não foi concluída.                     recarga_falha                                               3
                                                     recarga_pendente                                            1
Minha transferência ainda não apareceu.              saldo_não_atualizado_após_transferência_bancária            1
                                                     transferência_pendente                                      1
Onde encontro o PIN do meu cartão?                   obter_cartão_físico                                         1
                                                     pin_bloqueado                                               1
Parece que minha recarga foi cancelada.              recarga_cancelada                                           2
                                                     recarga_falha                                               1
Por que meu pagamento foi recusado?                  pagamento_do_cartão_recusado                                2
                                                     transferência_recusada                                      1
Por que minha recarga não foi concluída?             recarga_cancelada                                           2
                                                     recarga_falha                                               1
                                                     recarga_pendente                                            2
Por que minha transferência ainda não foi concluída? transferência_falha                                         1
                                                     transferência_pendente                                      1
Por que não consigo fazer uma transferência?         beneficiário_não_permitido                                  1
                                                     transferência_falha                                         2
Posso trocar moedas?                                 suporte_moeda_fiduciária                                    2
                                                     troca_via_aplicativo                                        1
Quais são as taxas para recargas?                    recarga_por_transferência_bancária_cobrança                 1
                                                     recarregar_por_cartão_cobrança                              2
Qual é a sua taxa de câmbio?                         cobrança_de_câmbio                                          1
                                                     taxa_de_câmbio                                              1
Qual é a taxa de câmbio atual?                       cobrança_de_câmbio                                          1
                                                     taxa_de_câmbio                                              2
Qual é a taxa de câmbio?                             cobrança_de_câmbio                                          5
                                                     taxa_de_câmbio                                              1
Quando a transação será concluída?                   pagamento_com_cartão_pendente                               1
                                                     transferência_não_recebida_pelo_destinatário                1
Quando minha transferência será concluída?           saldo_não_atualizado_após

In [64]:
# Removendo esses casos
df = df[~df['frase'].isin(conflitos)]
etapas['Frases com rótulos conflitantes'] = len(df)

# Conferência
print(f"Conferindo\nFrases com mais de um rótulo: {(df.groupby('frase')['rotulo'].nunique() > 1).sum()}")

Conferindo
Frases com mais de um rótulo: 0


In [65]:
print('Duplicadas:', df.duplicated(subset=['frase']).sum())
df = df.drop_duplicates(subset=['frase'])
etapas['Duplicatas exatas'] = len(df)
print('Linhas restantes:', len(df))

Duplicadas: 424
Linhas restantes: 12603


##### Duplicatas ignorando maiúsculas e pontuação

In [66]:
# Exemplo
df[df['frase'].str.contains('omo faço para recarregar meu cartão')]

,frase,rotulo
4252,Como faço para recarregar meu cartão?,recarregar_por_cartão
8013,Como faço para recarregar meu cartão com um cheque?,recarregar_por_dinheiro_ou_cheque
9693,como faço para recarregar meu cartão.,apple_pay_ou_google_pay


`Como faço para recarregar meu cartão?` e `como faço para recarregar meu cartão.` são a mesma frase, mas com diferenças de letra maiúscula e pontuação. Portanto, também excluo esses casos.

In [67]:
# temp: frase em minúsculas, sem pontuação e sem espaços extras
df['temp'] = (df['frase'].str.lower().str.replace(r'[^\w\s]', '', regex=True).str.split().str.join(' '))


n_rotulos = df.groupby('temp')['rotulo'].nunique()
conflitos_norm = n_rotulos[n_rotulos > 1].index
print(f'Conflitos: {len(conflitos_norm)} frases')
display(df[df['temp'].isin(conflitos_norm)].sort_values('temp')[['frase', 'rotulo']])

df = df[~df['temp'].isin(conflitos_norm)]
etapas['Conflitos (ignorando maiúsculas e pontuação)'] = len(df)

# Duplicatas que só aparecem depois de normalizar
print('Duplicadas:', df['temp'].duplicated().sum())
df = df.drop_duplicates(subset='temp').drop(columns='temp')
etapas['Duplicatas (ignorando maiúsculas e pontuação)'] = len(df)
print('Linhas restantes:', len(df))

Conflitos: 6 frases


,frase,rotulo
4252,Como faço para recarregar meu cartão?,recarregar_por_cartão
9693,como faço para recarregar meu cartão.,apple_pay_ou_google_pay
11200,Eu preciso verificar minha identidade?,por_que_verificar_identidade
13015,Eu preciso verificar minha identidade.,verificar_minha_identidade
3553,Há um pagamento que não reconheço.,pagamento_cartão_não_reconhecido
4792,Há um pagamento que não reconheço!,pagamento_por_débito_direto_não_reconhecido
6053,Minha transferência está pendente?,transferência_pendente
12714,Minha transferência está pendente.,saldo_não_atualizado_após_transferência_bancária
11228,O que eu preciso para verificar minha identidade?,incapaz_de_verificar_identidade
13014,O que eu preciso para verificar minha identidade.,verificar_minha_identidade


Duplicadas: 53
Linhas restantes: 12538


#### 1.5 Resumo do pré-processamento

In [68]:
resumo = pd.Series(etapas, name='linhas restantes').to_frame()
resumo['removidas'] = (-resumo['linhas restantes'].diff()).fillna(0).astype(int)
display(resumo)

print(f"Total removido: {etapas['Dataset original'] - len(df)} linhas")
print(f"Classes: {df['rotulo'].nunique()}")

,linhas restantes,removidas
Dataset original,13085,0
Valores nulos,13083,2
Limpeza da tradução,13081,2
"Correção de ""abastecer"" e frase vazia",13080,1
Frases com rótulos conflitantes,13027,53
Duplicatas exatas,12603,424
Conflitos (ignorando maiúsculas e pontuação),12591,12
Duplicatas (ignorando maiúsculas e pontuação),12538,53


Total removido: 547 linhas
Classes: 77


In [69]:
# Salvar CSV
df.to_csv('../data/processed/df_01.csv', index=False)